In [ ]:
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

In [ ]:
fig_dir="outputs/figures/work_queue"
os.makedirs(fig_dir, exist_ok=True)

files = glob.glob("outputs/runs/**/makeflowlog.csv", recursive=True)
paths = [os.path.dirname(p) for p in files]

In [ ]:
fig, ax = plt.subplots()
for path in paths:
    df = pd.read_csv(f'{path}/wq_log.csv')

    if df["tasks_done"].iloc[-1] < 72:
        continue

    ax.plot(df["normalized_datetime"], df["tasks_done"], label=str(df.num_cores.iloc[0]))

    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Jobs Complete")
    ax.set_title(f"All Runs Together")
plt.savefig(f"{fig_dir}/all_runs_combined.pdf", dpi=300, bbox_inches='tight')
plt.savefig(f"{fig_dir}/all_runs_combined.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
for num_cores in [4, 8, 16, 32, 64]:
    fig, ax = plt.subplots()
    for path in paths:
        if int(path.split("/")[3]) != 1:
            continue

        df = pd.read_csv(f'{path}/wq_log.csv')

        if df["num_cores"].iloc[0] != num_cores:
            continue
        if df["tasks_done"].iloc[-1] < 72:
            continue

        df.drop(df.index[-1], inplace=True)

        ax.plot(df["normalized_datetime"], df["workers_busy"], label=str(df.num_cores.iloc[0]))
        # ax.plot(df["normalized_datetime"], df["workers_connected"], label=str(df.num_cores.iloc[0]))
        # ax.plot(df["normalized_datetime"], df["workers_able"], label=str(df.num_cores.iloc[0]))

        ax.set_xlim(0, 35000)
        ax.set_ylim(0, 75)

        ax.set_xlabel("Time (s)", fontsize=14)
        ax.set_ylabel("Workers Connected", fontsize=14)
        ax.set_title(f"{num_cores} Core Workers", fontsize=16)
        ax.grid(True, linestyle="--", alpha=0.6)

    plt.savefig(f"{fig_dir}/workers vs. time {num_cores} cores.pdf", dpi=300, bbox_inches='tight')
    plt.savefig(f"{fig_dir}/workers vs. time {num_cores} cores.png", dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 8))
# fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=True)
axes = axes.flatten()

for ax, num_cores in zip(axes, [4, 8, 16, 32, 64]):
    for path in paths:
        # must be first run of session.
        if int(path.split("/")[3]) != 1:
            continue

        df = pd.read_csv(f"{path}/wq_log.csv")

        if df["num_cores"].iloc[0] != num_cores:
            continue
        if df["tasks_done"].iloc[-1] < 72:
            continue

        df = df.iloc[:-1]  # drop last row

        ax.plot(df["normalized_datetime"], df["workers_busy"], label=str(df.num_cores.iloc[0]))
        # ax.plot(df["normalized_datetime"], df["workers_connected"], label=str(df.num_cores.iloc[0]))
        # ax.plot(df["normalized_datetime"], df["workers_able"], label=str(df.num_cores.iloc[0]))

    ax.set_title(f"{num_cores} Cores")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Workers Busy")
    ax.set_xlim(0, 33000)
    ax.set_ylim(0, 75)
    ax.grid(True, linestyle="--", alpha=0.6)

# Remove the unused 6th subplot
fig.delaxes(axes[-1])

fig.suptitle("Workers Busy vs. Time", fontsize=16)
fig.tight_layout(rect=(0, 0, 1, 0.96))
plt.savefig(f"{fig_dir}/workers_vs_time_all_cores.pdf", dpi=300, bbox_inches="tight")
plt.savefig(f"{fig_dir}/workers_vs_time_all_cores.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(figsize=(10, 8))
# fig, axes = plt.subplots(figsize=(15, 8))
execution_times = pd.DataFrame(columns=["num_cores", "max_workers", "execution_time"])
cores_map = {
    4 : "red",
    8 : "blue",
    16: "green",
    32: "purple",
    64: "orange",
}

for path in paths:
    # must be first run of session.
    if int(path.split("/")[3]) != 1:
        continue
    df = pd.read_csv(f"{path}/wq_log.csv")
    sims_df = pd.read_csv(f"{path}/of_sims.csv")
    if df["tasks_done"].iloc[-1] < 72:
        continue

    df = df.iloc[:-1]  # drop last row

    
    max_workers = df[df["workers_connected"] == df["workers_connected"].max()].iloc[0]["normalized_datetime"]
    half_max_workers = df[df["workers_connected"] >= df["workers_connected"].max()/2].iloc[0]["normalized_datetime"]
    mean_exec_time = sims_df["execution_time"].mean()

    num_cores = df["num_cores"].iloc[0]
    new_row = pd.DataFrame({
        "num_cores": [num_cores],
        "max_workers": [max_workers],
        "execution_time": [mean_exec_time],
    })
    execution_times = pd.concat([execution_times, new_row], ignore_index=True)

    plt.scatter(half_max_workers, mean_exec_time, color=cores_map[num_cores], s=40)
    plt.scatter(max_workers, mean_exec_time,      color=cores_map[num_cores], s=90)
    plt.plot([half_max_workers, max_workers], [mean_exec_time, mean_exec_time], color=cores_map[num_cores], linestyle="dashed")
    # plt.plot([half_max_workers, max_workers], [mean_exec_time, mean_exec_time], color=cores_map[num_cores], linewidth=1.5, alpha=0.7)



plt.title(f"Wait Time vs. Execution Time (s)", fontsize=20)
plt.xlabel("Queuing Time (s)", fontsize=18)
plt.ylabel("Average Task Execution Time (s)", fontsize=18)
ax.tick_params(axis='both', labelsize=12)

plt.ylim(0)
plt.xlim(0)
axes.tick_params(axis="both", labelsize=14)
plt.grid(True, linestyle="--", alpha=0.6)

legend_handles = [
    Line2D(
        [0], [0],
        marker='o',
        linestyle='None',
        markerfacecolor=color,
        markeredgecolor=color,
        markersize=10,
        label=f"{cores}"
    )
    for cores, color in cores_map.items()
] + [
    Line2D([0], [0], marker='o', color='gray', linestyle='-', markersize=8,
           label="Half Max → Max Workers")
]

plt.legend(handles=legend_handles, title="CPU Cores")


plt.savefig(f"{fig_dir}/wait time vs. execution time.pdf", dpi=300, bbox_inches="tight")
plt.savefig(f"{fig_dir}/wait time vs. execution time.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
for num_cores in [4, 8, 16, 32, 64]:
    fig, axes = plt.subplots(2, 2, figsize=(15, 8))
    axes = axes.flatten()

    idx = 0
    for ax in axes:
        for i in range(idx, len(paths)):
            path = paths[i]
            idx += 1
            # must be first run of session.
            if int(path.split("/")[3]) != 1:
                continue

            df = pd.read_csv(f"{path}/wq_log.csv")

            if df["num_cores"].iloc[0] != num_cores:
                continue
            if df["tasks_done"].iloc[-1] < 72:
                continue

            df = df.iloc[:-1]  # drop last row

            ax.plot(df["normalized_datetime"], df["workers_connected"], linestyle="solid",  label="Workers Connected")
            ax.plot(df["normalized_datetime"], df["workers_busy"],      linestyle="dashed", label="Workers Running")
            ax.plot(df["normalized_datetime"], df["tasks_running"],     linestyle="dashdot",label="Tasks Running")
            ax.plot(df["normalized_datetime"], df["tasks_done"],        linestyle="dotted",label="Tasks Done")
            break

        ax.set_title(f"{num_cores} Cores")
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Workers Busy")
        ax.grid(True, linestyle="--", alpha=0.6)
        

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels,
            loc="upper center",
            bbox_to_anchor=(0.5, 0.95),
            ncol=4)
    fig.suptitle("Workers Busy vs. Time", fontsize=16)
    fig.tight_layout(rect=(0, 0, 1, 0.96))

    plt.savefig(f"{fig_dir}/workers and tasks combined {num_cores} cores.pdf", dpi=300, bbox_inches="tight")
    plt.savefig(f"{fig_dir}/workers and tasks combined {num_cores} cores.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
twin_axes = []

idx = 0

for ax_idx, num_cores in enumerate([4, 8, 16, 32, 64]):
    ax = axes[ax_idx]
    ax2 = ax.twinx()
    twin_axes.append(ax2)

    for i in range(idx, len(paths)):
        path = paths[i]
        if int(path.split("/")[3]) != 1:
            continue

        df = pd.read_csv(f"{path}/wq_log.csv")
        if df["num_cores"].iloc[0] != num_cores:
            continue
        if df["tasks_done"].iloc[-1] < 72:
            continue

        df = df.iloc[:-1]

        # ax.plot(df["normalized_datetime"], df["workers_connected"], linestyle="solid",  label="Workers Connected")
        # ax.plot(df["normalized_datetime"], df["workers_busy"],      linestyle="dashed", label="Workers Running")
        ax.plot(df["normalized_datetime"], df["tasks_running"],     linestyle="dashdot",label="Tasks Running")
        ax.plot(df["normalized_datetime"], df["tasks_done"],        linestyle="dotted", label="Tasks Done")
        ax2.plot(df["normalized_datetime"], df["total_cores"],      linestyle="dotted", label="Available Cores", color="gray")
        ax2.axhline(72*df["num_cores"].iloc[0],                     linestyle="dashed",   label="Requested Cores", color="red")
        break

    ax.set_title(f"{num_cores} Cores")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Workers Busy")
    ax.grid(True, linestyle="--", alpha=0.6)
    ax2.set_ylabel("Total Cores", color="tab:gray")
    ax2.tick_params(axis="y", labelcolor="tab:gray")

fig.delaxes(axes[-1])

handles1, labels1 = axes[0].get_legend_handles_labels()
handles2, labels2 = twin_axes[0].get_legend_handles_labels()
fig.legend(handles1 + handles2, labels1 + labels2,
           loc="upper center", bbox_to_anchor=(0.5, 0.95), ncol=5)

fig.suptitle("Workers Busy vs. Time", fontsize=16)
fig.tight_layout(rect=(0, 0, 1, 0.96))
plt.savefig(f"{fig_dir}/workers and tasks combined.pdf", dpi=300, bbox_inches="tight")
plt.savefig(f"{fig_dir}/workers and tasks combined.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

core_counts = [4, 8, 16, 32, 64]

for idx, num_cores in enumerate(core_counts):
    ax = axes[idx]  # <-- FIX 1: Assign the specific subplot axis for this iteration
    
    for i in range(len(paths)):
        path = paths[i]
        if int(path.split("/")[3]) != 1:
            continue

        df = pd.read_csv(f"{path}/wq_log.csv")
        if df["num_cores"].iloc[0] != num_cores:
            continue
        if df["tasks_done"].iloc[-1] < 72:
            continue

        df = df.iloc[:-1]

        ax.plot(df["normalized_datetime"], df["tasks_running"], label="Tasks Running")
        ax.plot(df["normalized_datetime"], df["tasks_done"],    label="Tasks Done")
        ax.plot(df["normalized_datetime"], df["total_cores"],   label="Total Cores")
        ax.axhline(72 * df["num_cores"].iloc[0],                label="Requested Cores", color="red", linestyle="--")

        break

    # Formatting each individual subplot
    ax.set_title(f"{num_cores} Core Workers")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Number of Cores")
    # ax.set_xlim(0, 33000)
    # ax.set_ylim(0, 5000)
    # ax.set_yscale("log")
    ax.grid(True, linestyle="--", alpha=0.6)
    
# Remove the 6th unused subplot (since you only have 5 core configurations)
fig.delaxes(axes[-1])

# <-- FIX 2: Automatically extract legend handles and labels from the first populated subplot
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.94), ncol=2)

fig.suptitle("Resource Available vs. Time", fontsize=16)
fig.tight_layout(rect=(0, 0, 1, 0.93))
plt.show()

In [ ]:
core_counts = [4, 8, 16, 32, 64]

for idx, num_cores in enumerate(core_counts):
    fig, axes = plt.subplots()
    # fig, axes = plt.subplots(figsize=(15, 8))
    for i in range(len(paths)):
        path = paths[i]
        if int(path.split("/")[3]) != 1:
            continue

        df = pd.read_csv(f"{path}/wq_log.csv")
        if df["num_cores"].iloc[0] != num_cores:
            continue
        if df["tasks_done"].iloc[-1] < 72:
            continue

        df = df.iloc[:-1]

        requested_cores = 72 * df["num_cores"].iloc[0]
        df["percent_cores_of_requested"] = (df["total_cores"]/requested_cores)*100
        df["percent_tasks_done"] = (df["tasks_done"]/74)*100
        df["percent_tasks_running"] = (df["tasks_running"]/74)*100

        plt.plot(df["normalized_datetime"], df["percent_tasks_running"],        label="Tasks Running")
        plt.plot(df["normalized_datetime"], df["percent_tasks_done"],           label="Tasks Completed")
        plt.plot(df["normalized_datetime"], df["percent_cores_of_requested"],   label="Cores Available")
        break

    # Formatting each individual subplot
    plt.title(f"{num_cores} Core Workers", fontsize=16)
    plt.xlabel("Time (s)", fontsize=14)
    plt.ylabel("Percentage", fontsize=14)
    plt.xlim(0, 33000)
    plt.ylim(0, 110)
    # plt.yscale("log")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend(loc="best")
    # plt.legend(loc="best", ncol=2)
    # plt.legend(loc="upper left", bbox_to_anchor=(0, 0.95), ncol=2)
    
plt.show()